In [ ]:
# # 05.2 — Estimativa Causal: IPTW / AIPW + Heterogeneidade + E-value
# **Depende de:** 05.1 executado antes (artefatos em out/artifacts_causal/)
# **Salva em:** out/artifacts_causal/ e out/05.2_estimativa/

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm as _norm

warnings.filterwarnings("ignore")

ROOT        = Path("..").resolve()
OUT_DIR     = ROOT / "out" / "05.2_estimativa"
ART_CAU_DIR = ROOT / "out" / "artifacts_causal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Carregar artefatos do 05.1 ────────────────────────────────────────────────
import pickle

with open(ART_CAU_DIR / "cohort.pkl", "rb") as f:
    _art = pickle.load(f)

df_trim       = _art["df_trim"]
df_analise    = _art["df_analise"]
X_ps          = _art["X_ps"]
X_ps_trim     = _art["X_ps_trim"]
T             = _art["T"]
Y             = _art["Y"]
T_trim        = _art["T_trim"]
Y_trim        = _art["Y_trim"]
ps_gbm_full   = _art["ps_gbm"]
ps_logr_full  = _art["ps_logr"]
ps_trim       = _art["ps_trim"]
ps_logr_trim  = _art["ps_logr_trim"]
mask_overlap  = _art["mask_overlap"]
w_clip        = _art["w_iptw"]
w_logr_clip   = _art["w_iptw_logr"]
CONFUNDIDORES = _art["CONFUNDIDORES"]
df_dur        = _art["df_dur"]
eps           = _art["eps"]
TRIM_LO       = _art["TRIM_LO"]
TRIM_HI       = _art["TRIM_HI"]

print(f"Artefatos carregados de: {ART_CAU_DIR}")
print(f"  Coorte trimada: {len(df_trim):,}  |  T=1: {T_trim.sum():,}  |  Y=1: {Y_trim.sum():,}")

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "fp_ate_lbl_fmt":    "ATE geral ({ate:+.2f} pp)",
        "fp_xlabel":         "CATE (pp) com IC95%",
        "fp_title":          "Forest Plot — Heterogeneidade de Efeito (CATE) por Subgrupo",
        "dur_sem_lbl_fmt":   "Sem documentação (n={n:,})",
        "dur_com_lbl_fmt":   "Com documentação (n={n:,})",
        "dur_xlabel":        "Duração (horas)",
        "dur_ylabel":        "Densidade",
        "dur_title":         "Distribuição de Duração por Documentação",
        "dur_ate_lbl_fmt":   "ATE geral ({ate:+.1f}h)",
        "dur_seg_xlabel":    "Horas extras (ATE)",
        "dur_seg_title":     "Efeito por Segmento CID",
    },
    "en": {
        "fp_ate_lbl_fmt":    "Overall ATE ({ate:+.2f} pp)",
        "fp_xlabel":         "CATE (pp) with 95% CI",
        "fp_title":          "Forest Plot — Effect Heterogeneity (CATE) by Subgroup",
        "dur_sem_lbl_fmt":   "No documentation (n={n:,})",
        "dur_com_lbl_fmt":   "With documentation (n={n:,})",
        "dur_xlabel":        "Duration (hours)",
        "dur_ylabel":        "Density",
        "dur_title":         "Duration Distribution by Documentation",
        "dur_ate_lbl_fmt":   "Overall ATE ({ate:+.1f}h)",
        "dur_seg_xlabel":    "Extra hours (ATE)",
        "dur_seg_title":     "Effect by ICD-10 Segment",
    },
}
T = _T[LANG]

## 1. IPTW ATE

In [ ]:
eps = 1e-6
p_t = T_trim.mean()

Y1_iptw = (Y_trim[T_trim == 1] * w_clip[T_trim == 1]).sum() / w_clip[T_trim == 1].sum()
Y0_iptw = (Y_trim[T_trim == 0] * w_clip[T_trim == 0]).sum() / w_clip[T_trim == 0].sum()
ate_iptw = Y1_iptw - Y0_iptw

print(f"\nIPTW — Estimativa ATE (Retorno 30d):")
print(f"  E[Y(1)] = {Y1_iptw:.4f}  ({Y1_iptw*100:.2f}%)")
print(f"  E[Y(0)] = {Y0_iptw:.4f}  ({Y0_iptw*100:.2f}%)")
print(f"  ATE     = {ate_iptw:+.4f}  ({ate_iptw*100:+.2f} pp)")

## 2. AIPW com Cross-Fitting (5-fold) — Estimador Duplamente Robusto

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold

# ── 2a. AIPW com cross-fitting (estimador sem viés de treino/teste) ───────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
score_1_cf = np.zeros(len(Y_trim))
score_0_cf = np.zeros(len(Y_trim))

for fold_i, (train_idx, test_idx) in enumerate(kf.split(X_ps_trim)):
    print(f"  Cross-fit fold {fold_i+1}/5 …", end=" ", flush=True)

    # Modelo PS no fold de treino
    ps_fold = GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
    )
    ps_fold.fit(X_ps_trim[train_idx], T_trim[train_idx])
    ps_test = ps_fold.predict_proba(X_ps_trim[test_idx])[:, 1]

    # Modelo de outcome no fold de treino
    X_out_fold = np.hstack([X_ps_trim[train_idx], T_trim[train_idx].reshape(-1, 1)])
    out_fold = GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
    )
    out_fold.fit(X_out_fold, Y_trim[train_idx])

    # Predições contrafactuais no fold de teste
    X1_test = np.hstack([X_ps_trim[test_idx], np.ones((len(test_idx), 1))])
    X0_test = np.hstack([X_ps_trim[test_idx], np.zeros((len(test_idx), 1))])
    mu1_test = out_fold.predict_proba(X1_test)[:, 1]
    mu0_test = out_fold.predict_proba(X0_test)[:, 1]

    # Scores de influência (influence function) no fold de teste
    score_1_cf[test_idx] = (
        mu1_test
        + T_trim[test_idx] * (Y_trim[test_idx] - mu1_test) / (ps_test + eps)
    )
    score_0_cf[test_idx] = (
        mu0_test
        + (1 - T_trim[test_idx]) * (Y_trim[test_idx] - mu0_test) / (1 - ps_test + eps)
    )
    print("ok")

ate_aipw_cf = (score_1_cf - score_0_cf).mean()

# Bootstrap IC95% sobre os scores cross-fittados
rng = np.random.default_rng(42)
boot_cf = [
    (score_1_cf[rng.integers(0, len(Y_trim), len(Y_trim))] -
     score_0_cf[rng.integers(0, len(Y_trim), len(Y_trim))]).mean()
    for _ in range(500)
]
ic_lo_cf, ic_hi_cf = np.percentile(boot_cf, [2.5, 97.5])
se_cf = np.std(boot_cf)
pval_cf = 2 * (1 - _norm.cdf(abs(ate_aipw_cf / se_cf)))

print(f"\nAIPW com Cross-Fitting (5-fold) — estimativa primária:")
print(f"  ATE   = {ate_aipw_cf:+.4f}  ({ate_aipw_cf*100:+.2f} pp)")
print(f"  IC95% = [{ic_lo_cf:+.4f}, {ic_hi_cf:+.4f}]")
print(f"  SE    = {se_cf:.4f}")
print(f"  p     = {pval_cf:.4f}")

# ── 2b. AIPW padrão (sem cross-fitting — para comparação) ────────────────────
X_out = np.hstack([X_ps_trim, T_trim.reshape(-1, 1)])
out_model_std = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
out_model_std.fit(X_out, Y_trim)

X1_std = np.hstack([X_ps_trim, np.ones((len(T_trim), 1))])
X0_std = np.hstack([X_ps_trim, np.zeros((len(T_trim), 1))])
mu1_std = out_model_std.predict_proba(X1_std)[:, 1]
mu0_std = out_model_std.predict_proba(X0_std)[:, 1]

score_1_std = mu1_std + T_trim * (Y_trim - mu1_std) / (ps_trim + eps)
score_0_std = mu0_std + (1 - T_trim) * (Y_trim - mu0_std) / (1 - ps_trim + eps)
ate_aipw_std = (score_1_std - score_0_std).mean()

boot_std = [
    (score_1_std[rng.integers(0, len(Y_trim), len(Y_trim))] -
     score_0_std[rng.integers(0, len(Y_trim), len(Y_trim))]).mean()
    for _ in range(500)
]
ic_lo_std, ic_hi_std = np.percentile(boot_std, [2.5, 97.5])
se_std = np.std(boot_std)
pval_std = 2 * (1 - _norm.cdf(abs(ate_aipw_std / se_std)))

print(f"\nAIPW padrão (sem cross-fitting) — comparação:")
print(f"  ATE   = {ate_aipw_std:+.4f}  ({ate_aipw_std*100:+.2f} pp)")
print(f"  IC95% = [{ic_lo_std:+.4f}, {ic_hi_std:+.4f}]")
print(f"  SE    = {se_std:.4f}")
print(f"  p     = {pval_std:.4f}")

# Estimativa principal a ser reportada no paper: cross-fitting
ate_aipw  = ate_aipw_cf
ic_lo     = ic_lo_cf
ic_hi     = ic_hi_cf
se        = se_cf
pval      = pval_cf
score_1   = score_1_cf
score_0   = score_0_cf

## 3. CATE por Subgrupo com Bootstrap IC95% — Forest Plot

In [ ]:
subgrupos = {
    "HIPERTENSIVOS":   df_trim["segmento_cid"] == "HIPERTENSIVOS",
    "DIABETES_GEST":   df_trim["segmento_cid"] == "DIABETES_GEST",
    "PARTO_PREMATURO": df_trim["segmento_cid"] == "PARTO_PREMATURO",
    "Idade < 18":      df_trim["idade_anos"] < 18,
    "Idade >= 35":     df_trim["idade_anos"] >= 35,
    "Urgência/URI":    df_trim["tp_atendimento"] == "U",
    "Internação":      df_trim["tp_atendimento"] == "I",
}

N_BOOT_CATE = 500
rng_cate    = np.random.default_rng(42)
rows_het    = []

for nome_sub, mask_sub in subgrupos.items():
    mask_arr = mask_sub.values
    if mask_arr.sum() < 30:
        print(f"  Subgrupo {nome_sub}: n={mask_arr.sum()} < 30 — pulado")
        continue

    s1_sub   = score_1[mask_arr]
    s0_sub   = score_0[mask_arr]
    cate_est = (s1_sub - s0_sub).mean()

    boot_cate = []
    n_sub = mask_arr.sum()
    for _ in range(N_BOOT_CATE):
        idx = rng_cate.integers(0, n_sub, n_sub)
        boot_cate.append((s1_sub[idx] - s0_sub[idx]).mean())

    cate_lo, cate_hi = np.percentile(boot_cate, [2.5, 97.5])
    cate_se          = np.std(boot_cate)
    cate_pval        = 2 * (1 - _norm.cdf(abs(cate_est / (cate_se + 1e-12))))

    rows_het.append({
        "subgrupo":        nome_sub,
        "n":               int(n_sub),
        "prev_retorno_30d": round(df_trim[mask_sub]["Y"].mean() * 100, 1),
        "CATE_pp":         round(cate_est * 100, 2),
        "IC95_lo_pp":      round(cate_lo * 100, 2),
        "IC95_hi_pp":      round(cate_hi * 100, 2),
        "p_valor":         round(cate_pval, 4),
    })

het_df = pd.DataFrame(rows_het)
print("\nHeterogeneidade de efeito (CATE por subgrupo, Bootstrap IC95%):")
print(het_df.to_string(index=False))
het_df.to_csv(OUT_DIR / "cate_subgrupos.csv", index=False)

# Forest plot
fig, ax = plt.subplots(figsize=(10, max(5, len(het_df) * 0.65)))
y_pos   = np.arange(len(het_df))
labels  = het_df["subgrupo"].tolist()

cate_vals = het_df["CATE_pp"].values
lo_err    = cate_vals - het_df["IC95_lo_pp"].values
hi_err    = het_df["IC95_hi_pp"].values - cate_vals

colors = ["#d62728" if v > 0 else "#2ca02c" for v in cate_vals]

ax.errorbar(
    cate_vals, y_pos,
    xerr=[lo_err, hi_err],
    fmt="o", color="black", ecolor="gray",
    capsize=4, elinewidth=1.2, markerfacecolor="black", markersize=6,
)
for yi, ci, c in zip(y_pos, cate_vals, colors):
    ax.scatter([ci], [yi], color=c, zorder=5, s=50)

ax.axvline(0, color="black", lw=0.8, ls="-")
ax.axvline(ate_aipw * 100, color="blue", ls="--", lw=1.2,
           label=T["fp_ate_lbl_fmt"].format(ate=ate_aipw*100))
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel(T["fp_xlabel"])
ax.set_title(T["fp_title"])
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "forest_plot_cate.png", dpi=150)
plt.show()
plt.close()

## 4. E-value (VanderWeele & Ding 2017) — Robustez a Confundimento Não Medido

In [ ]:
def e_value_risk_diff(ate, baseline_risk, ic_bound=None):
    """
    Calcula o E-value para uma diferença de risco usando a fórmula de
    VanderWeele & Ding (2017, Annals of Internal Medicine).

    Converte a diferença de risco (ATE) em Risk Ratio (RR) aproximado,
    depois aplica: E = RR + sqrt(RR * (RR - 1))

    Para o limite do IC (mais conservador), usa o RR correspondente ao
    limite mais próximo de zero.

    Parâmetros
    ----------
    ate : float
        Diferença de risco estimada (AIPW ATE)
    baseline_risk : float
        Risco de base (estimativa de E[Y(0)] — contrafactual sem exposição)
    ic_bound : float, opcional
        Limite do IC95% mais próximo de nulo (usado para E-value do IC)

    Retorna
    -------
    ev : float
        E-value para a estimativa pontual
    ev_ic : float | None
        E-value para o limite do IC (None se ic_bound cruzar zero)
    rr_est : float
        RR aproximado utilizado no cálculo
    """
    p0 = float(baseline_risk)
    if p0 <= 0 or p0 >= 1:
        return np.nan, np.nan, np.nan

    p1 = p0 + float(ate)
    p1 = max(1e-6, min(1 - 1e-6, p1))

    rr_est = p1 / p0
    # Trabalha sempre com RR >= 1 (efeito protetivo ou de risco — mesmo fórmula)
    if rr_est < 1:
        rr_est = 1.0 / rr_est

    if rr_est <= 1.0:
        ev = 1.0
    else:
        ev = rr_est + np.sqrt(rr_est * (rr_est - 1))

    ev_ic = None
    if ic_bound is not None:
        p1_ic = p0 + float(ic_bound)
        p1_ic = max(1e-6, min(1 - 1e-6, p1_ic))
        rr_ic = p1_ic / p0
        if rr_ic < 1:
            rr_ic = 1.0 / rr_ic
        if rr_ic <= 1.0:
            ev_ic = 1.0
        else:
            ev_ic = rr_ic + np.sqrt(rr_ic * (rr_ic - 1))

    return ev, ev_ic, rr_est


# Risco de base: média de Y no grupo controle (T=0) após trimming
p0_baseline = Y_trim[T_trim == 0].mean()

# Limite do IC mais próximo de zero
ic_bound_conservative = ic_lo if abs(ic_lo) < abs(ic_hi) else ic_hi

ev_point, ev_ic, rr_approx = e_value_risk_diff(
    ate        = ate_aipw,
    baseline_risk = p0_baseline,
    ic_bound   = ic_bound_conservative,
)

print("\n" + "="*65)
print("E-VALUE (VanderWeele & Ding, 2017)")
print("="*65)
print(f"  Risco de base E[Y(0)]           : {p0_baseline:.4f} ({p0_baseline*100:.2f}%)")
print(f"  ATE estimado (AIPW CF)          : {ate_aipw:+.4f} ({ate_aipw*100:+.2f} pp)")
print(f"  RR aproximado                   : {rr_approx:.4f}")
print(f"  E-value (estimativa pontual)    : {ev_point:.3f}")
print(f"  E-value (limite IC95% conserv.) : {ev_ic:.3f}" if ev_ic else
      f"  E-value (IC)                    : IC cruza zero — sem efeito detectável")
print()
print("  Interpretação:")
print(f"  O efeito observado poderia ser explicado por um confundidor não medido")
print(f"  com RR >= {ev_point:.2f} tanto para a associação exposição-confundidor")
print(f"  quanto para a associação confundidor-desfecho.")
if ev_ic is not None:
    print(f"  Para invalidar o IC95%, seria necessário RR >= {ev_ic:.2f}.")

## 5. Análise de Sensibilidade — Limiares de Trimming

In [ ]:
TRIM_THRESHOLDS = [0.01, 0.05, 0.10]

sens_rows = []

for thr in TRIM_THRESHOLDS:
    mask_thr = (ps_gbm_full >= thr) & (ps_gbm_full <= (1 - thr))
    n_trim   = (~mask_thr).sum()
    n_ana    = mask_thr.sum()

    if n_ana < 50:
        print(f"  Threshold {thr}: n={n_ana} insuficiente — pulado")
        continue

    # Refit AIPW cross-fitted para este threshold
    X_thr   = X_ps[mask_thr]
    T_thr   = T[mask_thr]
    Y_thr   = Y[mask_thr]
    ps_thr  = ps_gbm_full[mask_thr]

    kf_s = KFold(n_splits=5, shuffle=True, random_state=42)
    s1_thr = np.zeros(n_ana)
    s0_thr = np.zeros(n_ana)

    for tr_i, te_i in kf_s.split(X_thr):
        ps_f = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X_thr[tr_i], T_thr[tr_i])
        ps_te = ps_f.predict_proba(X_thr[te_i])[:, 1]

        X_of = np.hstack([X_thr[tr_i], T_thr[tr_i].reshape(-1, 1)])
        out_f = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_of, Y_thr[tr_i])

        X1_te = np.hstack([X_thr[te_i], np.ones((len(te_i), 1))])
        X0_te = np.hstack([X_thr[te_i], np.zeros((len(te_i), 1))])
        mu1_te = out_f.predict_proba(X1_te)[:, 1]
        mu0_te = out_f.predict_proba(X0_te)[:, 1]

        s1_thr[te_i] = mu1_te + T_thr[te_i] * (Y_thr[te_i] - mu1_te) / (ps_te + eps)
        s0_thr[te_i] = mu0_te + (1 - T_thr[te_i]) * (Y_thr[te_i] - mu0_te) / (1 - ps_te + eps)

    ate_thr = (s1_thr - s0_thr).mean()
    rng_s   = np.random.default_rng(42)
    boot_s  = [
        (s1_thr[rng_s.integers(0, n_ana, n_ana)] -
         s0_thr[rng_s.integers(0, n_ana, n_ana)]).mean()
        for _ in range(500)
    ]
    lo_s, hi_s = np.percentile(boot_s, [2.5, 97.5])
    se_s       = np.std(boot_s)

    sens_rows.append({
        "threshold_lo":  thr,
        "threshold_hi":  1 - thr,
        "n_trimmed":     int(n_trim),
        "n_analise":     int(n_ana),
        "ATE_pp":        round(ate_thr * 100, 2),
        "IC95_lo_pp":    round(lo_s * 100, 2),
        "IC95_hi_pp":    round(hi_s * 100, 2),
        "SE":            round(se_s, 4),
    })
    print(f"  thr={thr}: n_trim={n_trim}  n_ana={n_ana}  ATE={ate_thr*100:+.2f}pp"
          f"  IC=[{lo_s*100:+.2f},{hi_s*100:+.2f}]")

sens_df = pd.DataFrame(sens_rows)
print(f"\nSensibilidade ao trimming:")
print(sens_df.to_string(index=False))
sens_df.to_csv(OUT_DIR / "sensibilidade_trimming.csv", index=False)

## 6. Segunda Pergunta Causal — Má Documentação → Duração da Internação (AIPW Contínuo)

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor

print(f"\nCoorte duração: {len(df_dur):,}")

T_doc  = df_dur["T_doc"].values
Y_dur  = df_dur["duracao_horas_aprox"].values
num_cols = [c for c in CONFUNDIDORES if c in df_dur.columns
            and pd.api.types.is_numeric_dtype(df_dur[c])]
cat_cols_d = [c for c in CONFUNDIDORES if c in df_dur.columns and c not in num_cols]

pre_dur = ColumnTransformer([
    ("num", Pipeline([
        ("imp", IterativeImputer(max_iter=10, random_state=42)),
        ("sc",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols_d),
], remainder="drop")

CONF_DUR = [c for c in CONFUNDIDORES if c in df_dur.columns]
X_dur    = pre_dur.fit_transform(df_dur[CONF_DUR])

ps_doc_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
ps_doc_model.fit(X_dur, T_doc)
ps_dur = ps_doc_model.predict_proba(X_dur)[:, 1]

# Trimming
mask_dur = (ps_dur >= TRIM_LO) & (ps_dur <= TRIM_HI)
print(f"Trimming: {(~mask_dur).sum()} obs. removidas")

X_dur_t  = X_dur[mask_dur]
T_dur_t  = T_doc[mask_dur]
Y_dur_t  = Y_dur[mask_dur]
ps_dur_t = ps_dur[mask_dur]

# Modelo de outcome contínuo (duração)
# Nota: para Y contínuo, o estimador de influência AIPW é:
#   score_1 = mu1 + T*(Y - mu1) / (ps + eps)
#   score_0 = mu0 + (1-T)*(Y - mu0) / (1 - ps + eps)
# (mesma forma funcional que no caso binário — válido para qualquer outcome)
X_out_dur = np.hstack([X_dur_t, T_dur_t.reshape(-1, 1)])
out_dur   = GradientBoostingRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
out_dur.fit(X_out_dur, Y_dur_t)

X1_dur = np.hstack([X_dur_t, np.ones((len(X_dur_t), 1))])
X0_dur = np.hstack([X_dur_t, np.zeros((len(X_dur_t), 1))])
mu1_dur = out_dur.predict(X1_dur)
mu0_dur = out_dur.predict(X0_dur)

sc1_dur = mu1_dur + T_dur_t * (Y_dur_t - mu1_dur) / (ps_dur_t + eps)
sc0_dur = mu0_dur + (1 - T_dur_t) * (Y_dur_t - mu0_dur) / (1 - ps_dur_t + eps)
ate_dur  = (sc1_dur - sc0_dur).mean()

# Bootstrap IC95%
rng_dur = np.random.default_rng(42)
boot_dur = [
    (sc1_dur[rng_dur.integers(0, len(Y_dur_t), len(Y_dur_t))] -
     sc0_dur[rng_dur.integers(0, len(Y_dur_t), len(Y_dur_t))]).mean()
    for _ in range(500)
]
ic_lo_dur, ic_hi_dur = np.percentile(boot_dur, [2.5, 97.5])
se_dur   = np.std(boot_dur)
pval_dur = 2 * (1 - _norm.cdf(abs(ate_dur / se_dur)))

print(f"\n{'='*55}")
print(f"EFEITO CAUSAL — Má documentação → Duração internação")
print(f"{'='*55}")
print(f"  ATE   = {ate_dur:+.2f} horas")
print(f"  IC95% = [{ic_lo_dur:+.2f}, {ic_hi_dur:+.2f}] horas")
print(f"  SE    = {se_dur:.2f}")
print(f"  p     = {pval_dur:.4f}")
print()
if ate_dur > 0:
    print(f"  Pacientes SEM documentação ficam em média {ate_dur:.1f}h A MAIS internadas")
    print(f"  (IC95% {ic_lo_dur:.1f}h – {ic_hi_dur:.1f}h), p={pval_dur:.4f}")
else:
    print("  Sem efeito significativo na direção esperada (MA documentação → maior duração)")

# Distribuição e efeito por segmento CID
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = np.linspace(0, 200, 40)
axes[0].hist(Y_dur_t[T_dur_t == 1], bins=bins, alpha=0.6, color="tomato",
             label=T["dur_sem_lbl_fmt"].format(n=T_dur_t.sum()), density=True)
axes[0].hist(Y_dur_t[T_dur_t == 0], bins=bins, alpha=0.6, color="steelblue",
             label=T["dur_com_lbl_fmt"].format(n=(1-T_dur_t).sum()), density=True)
axes[0].axvline(Y_dur_t[T_dur_t == 1].mean(), color="tomato",    ls="--", lw=1.5)
axes[0].axvline(Y_dur_t[T_dur_t == 0].mean(), color="steelblue", ls="--", lw=1.5)
axes[0].set_xlabel(T["dur_xlabel"])
axes[0].set_ylabel(T["dur_ylabel"])
axes[0].set_title(T["dur_title"])
axes[0].legend()
axes[0].grid(alpha=0.3)

# Efeito por segmento CID
rows_dur_sub = []
for seg_nome in df_dur["segmento_cid"].value_counts().index[:6]:
    mask_s = (df_dur["segmento_cid"] == seg_nome).values[mask_dur]
    if mask_s.sum() < 20:
        continue
    ate_s = (sc1_dur[mask_s] - sc0_dur[mask_s]).mean()
    rows_dur_sub.append({
        "segmento": seg_nome,
        "n":        int(mask_s.sum()),
        "ate_horas": round(ate_s, 2),
    })

dur_sub   = pd.DataFrame(rows_dur_sub).sort_values("ate_horas", ascending=False)
cores_d   = ["#d62728" if v > 0 else "#2ca02c" for v in dur_sub["ate_horas"]]
axes[1].barh(dur_sub["segmento"][::-1], dur_sub["ate_horas"][::-1], color=cores_d[::-1])
axes[1].axvline(0, color="black", lw=0.8)
axes[1].axvline(ate_dur, color="blue", ls="--", lw=1,
                label=T["dur_ate_lbl_fmt"].format(ate=ate_dur))
axes[1].set_xlabel(T["dur_seg_xlabel"])
axes[1].set_title(T["dur_seg_title"])
axes[1].legend()
axes[1].grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "duracao_documentacao.png", dpi=150)
plt.show()
plt.close()

dur_sub.to_csv(OUT_DIR / "causal_duracao_por_segmento.csv", index=False)

## 7. Tabela Resumo de Todas as Estimativas Causais (para o Paper)

In [ ]:
import json as _json

# p-valor da estimativa IPTW (bootstrap simples)
rng_iptw = np.random.default_rng(42)
boot_iptw = []
for _ in range(500):
    idx = rng_iptw.integers(0, len(Y_trim), len(Y_trim))
    y1_b = (Y_trim[idx][T_trim[idx] == 1] * w_clip[idx][T_trim[idx] == 1]).sum() / \
           (w_clip[idx][T_trim[idx] == 1].sum() + 1e-12)
    y0_b = (Y_trim[idx][T_trim[idx] == 0] * w_clip[idx][T_trim[idx] == 0]).sum() / \
           (w_clip[idx][T_trim[idx] == 0].sum() + 1e-12)
    boot_iptw.append(y1_b - y0_b)
se_iptw   = np.std(boot_iptw)
pval_iptw = 2 * (1 - _norm.cdf(abs(ate_iptw / se_iptw)))
ic_lo_iptw, ic_hi_iptw = np.percentile(boot_iptw, [2.5, 97.5])

summary_rows = [
    {
        "Pergunta":   "Documentação precoce → Retorno 30d",
        "Metodo":     "IPTW (ATE)",
        "N":          int(len(Y_trim)),
        "N_exposto":  int(T_trim.sum()),
        "ATE_pp":     round(ate_iptw * 100, 2),
        "IC95_lo_pp": round(ic_lo_iptw * 100, 2),
        "IC95_hi_pp": round(ic_hi_iptw * 100, 2),
        "SE":         round(se_iptw, 4),
        "p_valor":    round(pval_iptw, 4),
        "E_value":    None,
    },
    {
        "Pergunta":   "Documentação precoce → Retorno 30d",
        "Metodo":     "AIPW padrão",
        "N":          int(len(Y_trim)),
        "N_exposto":  int(T_trim.sum()),
        "ATE_pp":     round(ate_aipw_std * 100, 2),
        "IC95_lo_pp": round(ic_lo_std * 100, 2),
        "IC95_hi_pp": round(ic_hi_std * 100, 2),
        "SE":         round(se_std, 4),
        "p_valor":    round(pval_std, 4),
        "E_value":    None,
    },
    {
        "Pergunta":   "Documentação precoce → Retorno 30d",
        "Metodo":     "AIPW Cross-Fitting 5-fold (primário)",
        "N":          int(len(Y_trim)),
        "N_exposto":  int(T_trim.sum()),
        "ATE_pp":     round(ate_aipw * 100, 2),
        "IC95_lo_pp": round(ic_lo * 100, 2),
        "IC95_hi_pp": round(ic_hi * 100, 2),
        "SE":         round(se, 4),
        "p_valor":    round(pval, 4),
        "E_value":    round(ev_point, 3) if not np.isnan(ev_point) else None,
    },
    {
        "Pergunta":   "Má documentação → Duração internação (horas)",
        "Metodo":     "AIPW contínuo",
        "N":          int(len(Y_dur_t)),
        "N_exposto":  int(T_dur_t.sum()),
        "ATE_pp":     round(ate_dur, 2),
        "IC95_lo_pp": round(ic_lo_dur, 2),
        "IC95_hi_pp": round(ic_hi_dur, 2),
        "SE":         round(se_dur, 4),
        "p_valor":    round(pval_dur, 4),
        "E_value":    None,
    },
]

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*90)
print("TABELA RESUMO — TODAS AS ESTIMATIVAS CAUSAIS (para o paper)")
print("="*90)
print(summary_df.to_string(index=False))

summary_df.to_csv(OUT_DIR / "resumo_causal_completo.csv", index=False)
(OUT_DIR / "resumo_causal.json").write_text(
    _json.dumps(summary_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 8. Sensibilidade — Escolha do Modelo de PS (GBM vs LogR)

In [ ]:
# AIPW cross-fitting com PS LogR
kf_lr   = KFold(n_splits=5, shuffle=True, random_state=42)
s1_lr   = np.zeros(len(Y_trim))
s0_lr   = np.zeros(len(Y_trim))

from sklearn.linear_model import LogisticRegression

for tr_lr, te_lr in kf_lr.split(X_ps_trim):
    ps_lr_f = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    ps_lr_f.fit(X_ps_trim[tr_lr], T_trim[tr_lr])
    ps_lr_te = ps_lr_f.predict_proba(X_ps_trim[te_lr])[:, 1]

    X_of_lr = np.hstack([X_ps_trim[tr_lr], T_trim[tr_lr].reshape(-1, 1)])
    out_lr_f = GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
    )
    out_lr_f.fit(X_of_lr, Y_trim[tr_lr])

    X1_lr = np.hstack([X_ps_trim[te_lr], np.ones((len(te_lr), 1))])
    X0_lr = np.hstack([X_ps_trim[te_lr], np.zeros((len(te_lr), 1))])
    mu1_lr = out_lr_f.predict_proba(X1_lr)[:, 1]
    mu0_lr = out_lr_f.predict_proba(X0_lr)[:, 1]

    s1_lr[te_lr] = mu1_lr + T_trim[te_lr] * (Y_trim[te_lr] - mu1_lr) / (ps_lr_te + eps)
    s0_lr[te_lr] = mu0_lr + (1 - T_trim[te_lr]) * (Y_trim[te_lr] - mu0_lr) / (1 - ps_lr_te + eps)

ate_lr = (s1_lr - s0_lr).mean()
rng_lr = np.random.default_rng(42)
boot_lr = [
    (s1_lr[rng_lr.integers(0, len(Y_trim), len(Y_trim))] -
     s0_lr[rng_lr.integers(0, len(Y_trim), len(Y_trim))]).mean()
    for _ in range(500)
]
ic_lo_lr, ic_hi_lr = np.percentile(boot_lr, [2.5, 97.5])
se_lr   = np.std(boot_lr)
pval_lr = 2 * (1 - _norm.cdf(abs(ate_lr / se_lr)))

ps_model_rows = [
    {
        "metodo":      "AIPW CF (GBM propensity)",
        "ATE_pp":      round(ate_aipw * 100, 2),
        "IC95_lo_pp":  round(ic_lo * 100, 2),
        "IC95_hi_pp":  round(ic_hi * 100, 2),
        "SE":          round(se, 4),
        "p_valor":     round(pval, 4),
    },
    {
        "metodo":      "AIPW CF (LogR propensity)",
        "ATE_pp":      round(ate_lr * 100, 2),
        "IC95_lo_pp":  round(ic_lo_lr * 100, 2),
        "IC95_hi_pp":  round(ic_hi_lr * 100, 2),
        "SE":          round(se_lr, 4),
        "p_valor":     round(pval_lr, 4),
    },
]

ps_sens_df = pd.DataFrame(ps_model_rows)
print(f"\nSensibilidade — modelo de PS:")
print(ps_sens_df.to_string(index=False))
ps_sens_df.to_csv(OUT_DIR / "sensibilidade_ps_modelo.csv", index=False)

## Resumo dos Outputs

In [ ]:
print("\n" + "="*65)
print("05.2 — OUTPUTS GERADOS")
print("="*65)
print(f"  out/05.2_estimativa/")
print(f"    forest_plot_cate.png          — forest plot CATE por subgrupo")
print(f"    cate_subgrupos.csv            — CATE + IC95% por subgrupo")
print(f"    sensibilidade_trimming.csv    — sensibilidade ao limiar de trimming")
print(f"    sensibilidade_ps_modelo.csv   — sensibilidade à escolha do PS")
print(f"    duracao_documentacao.png      — distribuição e efeito por CID (2ª pergunta)")
print(f"    causal_duracao_por_segmento.csv")
print(f"    resumo_causal_completo.csv    — todas as estimativas (Tabela 3 paper)")
print(f"    resumo_causal.json            — idem em JSON")
print("="*65)
print(f"\nEstimativa primária (AIPW Cross-Fitting 5-fold):")
print(f"  ATE = {ate_aipw*100:+.2f} pp  IC95% [{ic_lo*100:+.2f}, {ic_hi*100:+.2f}]  p={pval:.4f}")
print(f"  E-value = {ev_point:.3f}" if not np.isnan(ev_point) else "  E-value = N/A")